# Prediction on unseen graph families

Fit GPT-4o-mini update rules on training questions and the eight training random graphs. Evaluate held-out questions on four graph families: fresh random graphs, the supplied low-rank graphs, and square and triangular lattices.

Each family reports **one-step** and **rollout** flip-and-class balanced accuracy. One-step prediction uses the observed previous state; rollout starts from the observed initial state and feeds predictions back for eight steps. Models are fitted once per question regime and reused across all four families.

Run all cells from `res/` or the repository root. The notebook reads the main consolidated runs and the episode files under `data/lowrank/`; it does not call a language-model API or overwrite fitted coupling files.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "res":
    ROOT = ROOT.parent
if not (ROOT / "data" / "clean_runs.json").is_file():
    raise FileNotFoundError("Run this notebook from the repository root or res/.")
sys.path.insert(0, str(ROOT / "res"))
sys.path.insert(0, str(ROOT))

from utils import (S_PREV, FIELD, D_POS, D_NEG, Logistic, load_data,
                   build_xy, two_stage_fit, discrete_rollout, fcba)
from lib.datagen.graph import make_lattice_J

DATA = ROOT / "data"
MODEL = "gpt-4o-mini"
REGIMES = ["subjective", "objective"]
FAMILIES = ["Random", "Low Rank", "Square", "Triangular"]
METHODS = ["Persistence", "Interaction-Free", "Mean-Field",
           "Discrete Update", "+ 3 Couplings"]
T = 8
runs, banks, P, GCLASS = load_data(DATA)


## Load the supplied low-rank episodes

Use episode files that actually exist. The manifests also list planned trajectories that are absent from the release. Question splits come from the shared question banks, not from filename prefixes. Every loaded low-rank episode must use a test question.

Graph-family labels come from the source directory and exact lattice matrices. An all-positive low-rank graph is still part of the Low Rank evaluation. The inventory below reports measured adjacency rank and actual question/replica coverage.

In [2]:
LOW_DIR = DATA / "lowrank" / MODEL / "ood_lowrank"
low_runs, low_classes, graph_inventory = {}, {}, []
for regime in REGIMES:
    folder = LOW_DIR / f"{regime}_rank_frustration"
    manifest = json.loads((folder / "manifest.json").read_text())
    files = sorted(p for p in folder.glob("*.json") if p.name != "manifest.json")
    if not files:
        raise FileNotFoundError(f"No episode files in {folder}")
    records, graph_matrices, identities = [], {}, set()
    for path in files:
        r = json.loads(path.read_text())
        meta = r["meta"]
        entry = manifest["replicas"][path.stem]
        info = banks[regime][r["statement"]]
        assert meta["model"] == MODEL and r["mode"] == regime
        assert meta["num_agents"] == 32 and meta["num_steps"] == T
        assert meta["k"] == 5 and meta["persona_order"] == "canonical_fixed"
        assert info["split"] == "test", f"Non-test low-rank episode: {path.name}"
        assert entry["qid"] == info["qid"] and entry["question"] == r["statement"]
        assert entry["rank"] == "LOW"
        identity = (info["qid"], entry["graph_id"], entry["repeat"])
        assert identity not in identities, f"Duplicate episode: {identity}"
        identities.add(identity)
        J = np.asarray(r["J"], dtype=float)
        spins = np.asarray(r["spins_history"])
        assert J.shape == (32, 32) and np.array_equal(J, J.T)
        assert np.all(np.diag(J) == 0) and np.isin(J, [-1, 0, 1]).all()
        assert spins.shape == (T + 1, 32) and np.isin(spins, [-1, 0, 1]).all()
        gid = entry["graph_id"]
        if gid in graph_matrices:
            assert np.array_equal(graph_matrices[gid], J)
        else:
            graph_matrices[gid] = J
            graph_inventory.append({
                "Regime": regime.capitalize(), "Graph": gid,
                "Adjacency rank": int(np.linalg.matrix_rank(J)),
                "Undirected edges": int(np.count_nonzero(J) // 2),
                "Negative edges": int(np.count_nonzero(J < 0) // 2),
                "Target frustration": entry["target_frust"]})
        records.append({"model": MODEL, "mode": regime,
                        "statement": r["statement"], "J": r["J"],
                        "spins_history": r["spins_history"],
                        "replica": entry["repeat"]})
    # Require a complete question × supplied graph × replica product.
    questions = {key[0] for key in identities}
    graph_ids = {key[1] for key in identities}
    replicas = {key[2] for key in identities}
    assert len(identities) == len(questions) * len(graph_ids) * len(replicas)
    assert len(graph_ids) == 6 and replicas == {0, 1}
    low_runs[regime] = records
    low_classes[regime] = {
        J.astype(np.int8).tobytes(): "lowrank" for J in graph_matrices.values()}
    print(f"{regime}: {len(records)} episodes, {len(questions)} test questions, "
          f"{len(graph_ids)} graphs, {len(replicas)} replicas per question/graph")

display(pd.DataFrame(graph_inventory).set_index(["Regime", "Graph"]))


subjective: 120 episodes, 10 test questions, 6 graphs, 2 replicas per question/graph
objective: 120 episodes, 10 test questions, 6 graphs, 2 replicas per question/graph


Adjacency rank  ...  Target frustration
Regime     Graph                               ...                    
Subjective rankLOW_frust00_s0              11  ...                 0.0
           rankLOW_frust00_s1              11  ...                 0.0
           rankLOW_frust10_s0              11  ...                 0.1
           rankLOW_frust10_s1              11  ...                 0.1
           rankLOW_frust20_s0              11  ...                 0.2
           rankLOW_frust20_s1              11  ...                 0.2
Objective  rankLOW_frust00_s0              11  ...                 0.0
           rankLOW_frust00_s1              11  ...                 0.0
           rankLOW_frust10_s0              11  ...                 0.1
           rankLOW_frust10_s1              11  ...                 0.1
           rankLOW_frust20_s0              11  ...                 0.2
           rankLOW_frust20_s1              11  ...                 0.2

[12 rows x 4 columns]

## Build the training and evaluation sets

Training excludes all lattices and low-rank graphs. Random evaluation uses the four fresh random graphs. Lattice evaluation uses all held-out questions; low-rank evaluation uses the held-out questions available in its directory. The coverage table makes any difference in question counts visible.

In [3]:
lattices = {name: make_lattice_J(4, 8, name.lower())
            for name in ("Square", "Triangular")}
datasets, coverage = {}, []
for regime in REGIMES:
    x, y, ep = build_xy(MODEL, regime, runs, banks, P, GCLASS)
    train = (ep["split"] == "train") & np.isin(ep["gcls"], ["seen", "train_only"])
    family = np.full(len(y), "", dtype=object)
    family[ep["gcls"] == "fresh"] = "Random"
    for name, J in lattices.items():
        family[np.all(ep["J"] == J, axis=(1, 2))] = name
    test = (ep["split"] == "test") & (family != "")
    lx, ly, lep = build_xy(MODEL, regime, low_runs[regime], banks, P,
                           low_classes[regime])
    train_graphs = {J.astype(np.int8).tobytes() for J in ep["J"][train]}
    eval_graphs = {J.astype(np.int8).tobytes() for J in ep["J"][test]}
    eval_graphs.update(low_classes[regime])
    assert len(train_graphs) == 8 and train_graphs.isdisjoint(eval_graphs)
    train_qids = set(ep["qid"][train])
    test_qids = set(ep["qid"][test]) | set(lep["qid"])
    assert train_qids.isdisjoint(test_qids)
    evaluation = {
        "x": np.concatenate([x[test], lx]),
        "y": np.concatenate([y[test], ly]),
        "J": np.concatenate([ep["J"][test], lep["J"]]),
        "family": np.concatenate([family[test], np.full(len(ly), "Low Rank")]),
        "qid": np.concatenate([ep["qid"][test], lep["qid"]]),
        "replica": np.concatenate([ep["rep"][test], lep["rep"]])}
    datasets[regime] = (x[train], y[train], evaluation)
    coverage.append({"Regime": regime.capitalize(), "Set": "Training random",
                     "Questions": len(train_qids), "Graphs": len(train_graphs),
                     "Replicas per question/graph": len(set(ep["rep"][train])),
                     "Episodes": int(train.sum())})
    for name in FAMILIES:
        mask = evaluation["family"] == name
        n_questions = len(set(evaluation["qid"][mask]))
        n_graphs = len({J.astype(np.int8).tobytes() for J in evaluation["J"][mask]})
        n_replicas = len(set(evaluation["replica"][mask]))
        assert mask.any() and mask.sum() == n_questions * n_graphs * n_replicas
        coverage.append({"Regime": regime.capitalize(), "Set": name,
                         "Questions": n_questions, "Graphs": n_graphs,
                         "Replicas per question/graph": n_replicas,
                         "Episodes": int(mask.sum())})

display(pd.DataFrame(coverage).set_index(["Regime", "Set"]))


Questions  ...  Episodes
Regime     Set                         ...          
Subjective Training random         10  ...       320
           Random                  10  ...       160
           Low Rank                10  ...       120
           Square                  10  ...        40
           Triangular              10  ...        40
Objective  Training random         20  ...       640
           Random                  20  ...       320
           Low Rank                10  ...       120
           Square                  20  ...        80
           Triangular              20  ...        80

[10 rows x 4 columns]

## Fit the update rules

The fitting and prediction steps use the same designs, optimizer, training rows, and deterministic rollout convention as `4_prediction.ipynb`. L2 penalizes the standardized field weights. The three-coupling fit estimates the unsigned contribution first and holds it fixed while estimating the signed contributions.

In [4]:
# the table methods; each returns (onestep, rollout) test predictions
def predict_all(x_tr, y_tr, x_te, J_te):
    rows = x_tr.reshape(-1, x_tr.shape[-1])          # pooled train transitions
    parsed = y_tr.reshape(-1) != 0                   # unparsed targets excluded
    y01 = (y_tr.reshape(-1)[parsed] > 0).astype(float)
    E, s0, phi = len(x_te), x_te[:, 0, :, S_PREV], x_te[:, 0, :, FIELD]
    out = {}

    # persistence: no change; rollout frozen at s(0)
    out["Persistence"] = (x_te[..., S_PREV].astype(int),
                          np.repeat(s0[:, None].astype(int), T, axis=1))

    # interaction-free: logistic on the static field only (state-independent)
    clf = Logistic().fit(rows[parsed][:, FIELD], y01)
    pred = clf.predict_spin(x_te[..., FIELD])
    out["Interaction-Free"] = (pred, pred)

    # mean-field (Curie-Weiss): [field | population mean s̄(t)]
    def with_sbar(xx):
        sbar = xx[..., S_PREV].mean(axis=-1)
        return np.concatenate([xx[..., FIELD],
                               np.broadcast_to(sbar[..., None, None],
                                               xx.shape[:-1] + (1,))], axis=-1)
    clf = Logistic().fit(with_sbar(x_tr).reshape(-1, 17)[parsed], y01)
    onestep = clf.predict_spin(with_sbar(x_te))
    s, rollout = s0.copy(), np.empty((E, T, 32), dtype=int)
    for t in range(T):
        sbar = np.broadcast_to(s.mean(axis=1)[:, None, None], (E, 32, 1))
        s = clf.predict_spin(np.concatenate([phi, sbar], axis=-1)).astype(float)
        rollout[:, t] = s
    out["Mean-Field"] = (onestep, rollout)

    # discrete update with 1 coupling [(J s)]
    pos, neg = rows[:, D_POS], rows[:, D_NEG]
    clf = Logistic().fit(np.concatenate(
        [rows[:, FIELD], (pos + neg)[:, None]], axis=-1)[parsed], y01)
    dr = (x_te[..., D_POS] + x_te[..., D_NEG])[..., None]
    onestep = clf.predict_spin(np.concatenate([x_te[..., FIELD], dr], axis=-1))
    out["Discrete Update"] = (onestep, discrete_rollout(phi, J_te, s0, clf.w, 1))

    # + 3 couplings [(J+ s) | (J- s) | (|J| s)]: collinear, so fit in two stages
    w = two_stage_fit(rows[:, FIELD], np.stack([pos, neg], axis=-1),
                      pos - neg, parsed, y01)
    dr = np.stack([x_te[..., D_POS], x_te[..., D_NEG],
                   x_te[..., D_POS] - x_te[..., D_NEG]], axis=-1)
    onestep = np.where(np.concatenate([x_te[..., FIELD], dr], axis=-1) @ w > 0, 1, -1)
    out["+ 3 Couplings"] = (onestep, discrete_rollout(phi, J_te, s0, w, 3))
    return out

## Score predictions

Balanced accuracy averages the four groups defined by flip/stay and next-state sign. The rollout score uses the observed transition groups, even though rollout predictions feed back their own states. Missing previous or next spins are excluded by the shared scoring function. Each family pools its available transitions; all four scoring groups must be present.

In [5]:
results = {}
for regime in REGIMES:
    x_train, y_train, evaluation = datasets[regime]
    predictions = predict_all(x_train, y_train, evaluation["x"], evaluation["J"])
    y_test = evaluation["y"]
    s0 = evaluation["x"][:, 0, :, S_PREV]
    observed_previous = evaluation["x"][..., S_PREV]
    for family in FAMILIES:
        mask = evaluation["family"] == family
        defined = mask[:, None, None] & (y_test != 0) & (observed_previous != 0)
        for flip in (False, True):
            for sign in (-1, 1):
                assert (defined & ((y_test != observed_previous) == flip)
                        & (y_test == sign)).any(), f"Empty scoring group: {regime}/{family}"
        for method in METHODS:
            one_step, rollout = predictions[method]
            results[regime, family, method] = (
                fcba(one_step, y_test, s0, mask), fcba(rollout, y_test, s0, mask))
    print(f"{regime}: fitted on {len(y_train)} episodes; "
          f"evaluated {len(y_test)} episodes across {len(FAMILIES)} graph families")

columns = pd.MultiIndex.from_product(
    [[regime.capitalize() for regime in REGIMES], FAMILIES, ["One-step", "Rollout"]],
    names=["Questions", "Graph family", "Prediction"])
scores = pd.DataFrame(
    [[score for regime in REGIMES for family in FAMILIES
      for score in results[regime, family, method]] for method in METHODS],
    index=pd.Index(METHODS, name="Method"), columns=columns)
display(scores.style.format("{:.1f}").highlight_max(axis=0, props="font-weight: bold"))


subjective: fitted on 320 episodes; evaluated 360 episodes across 4 graph families
objective: fitted on 640 episodes; evaluated 600 episodes across 4 graph families


## LaTeX export

The export reports the same scores to one decimal. Each column's highest displayed value is bold, including ties.

In [6]:
lines = [r"\begin{table}[t]", r"\centering", r"\small",
         r"\begin{adjustbox}{max width=\textwidth}",
         r"\begin{tabular}{l " + "cc " * 8 + "}", r"\toprule",
         r" & \multicolumn{8}{c}{Subjective} & \multicolumn{8}{c}{Objective} \\",
         r"\cmidrule(lr){2-9}\cmidrule(lr){10-17}",
         " & " + " & ".join(rf"\multicolumn{{2}}{{c}}{{{family}}}"
                            for _ in REGIMES for family in FAMILIES) + r" \\",
         "Method & " + " & ".join(["1-step & Rollout"] * 8) + r" \\",
         r"\midrule"]
rounded = scores.round(1)
for method in METHODS:
    values = []
    for column in scores.columns:
        value = rounded.loc[method, column]
        text = f"{value:.1f}"
        if value == rounded[column].max():
            text = rf"\textbf{{{text}}}"
        values.append(text)
    lines.append(method + " & " + " & ".join(values) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{adjustbox}",
          r"\caption{\textit{Generalization to unseen graph families.} "
          r"GPT-4o-mini one-step and rollout flip-and-class balanced accuracy "
          r"on held-out questions.}", r"\end{table}"]
print("\n".join(lines))

\begin{table}[t]
\centering
\small
\begin{adjustbox}{max width=\textwidth}
\begin{tabular}{l cc cc cc cc cc cc cc cc }
\toprule
 & \multicolumn{8}{c}{Subjective} & \multicolumn{8}{c}{Objective} \\
\cmidrule(lr){2-9}\cmidrule(lr){10-17}
 & \multicolumn{2}{c}{Random} & \multicolumn{2}{c}{Low Rank} & \multicolumn{2}{c}{Square} & \multicolumn{2}{c}{Triangular} & \multicolumn{2}{c}{Random} & \multicolumn{2}{c}{Low Rank} & \multicolumn{2}{c}{Square} & \multicolumn{2}{c}{Triangular} \\
Method & 1-step & Rollout & 1-step & Rollout & 1-step & Rollout & 1-step & Rollout & 1-step & Rollout & 1-step & Rollout & 1-step & Rollout & 1-step & Rollout \\
\midrule
Persistence & 50.0 & 64.0 & 50.0 & 56.9 & 50.0 & 65.2 & 50.0 & 57.4 & 50.0 & 55.0 & 50.0 & 53.1 & 50.0 & 55.1 & 50.0 & 47.0 \\
Interaction-Free & 48.6 & 48.6 & 46.2 & 46.2 & 46.5 & 46.5 & 47.2 & 47.2 & 49.3 & 49.3 & 43.4 & 43.4 & 46.8 & 46.8 & 49.6 & 49.6 \\
Mean-Field & 59.3 & 57.9 & 58.0 & 65.4 & 65.3 & 62.9 & 64.9 & 63.0 & 65.3 & 55.5 & 51.